# Fetch Data

Uses database: `fixed_tiktok_data_donation_dec_2025_cleaned_dataset`

Changes from previous version:
- Uses `video_watches_with_video_metadata` instead of `video_watches` + `video_metadata` separately
- Filters `phase2_flattened_sessions` by `cleaned_data_set_prolific_ids`

In [1]:
%pip install psycopg2-binary pandas scipy matplotlib lifelines


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2
import psycopg2.extras
import pandas as pd
import numpy as np
import warnings

pd.set_option('display.float_format', lambda x: '%.3f' % x)
warnings.filterwarnings("ignore")

In [ ]:
HOST = 'localhost'
DBNAME = 'fixed_tiktok_data_donation_dec_2025_cleaned_dataset'
USER = 'er3752'
PASSWORD = '!' 
PORT = 5432
DBAuthorize = "host=%s dbname=%s user=%s password=%s port=%s" % (HOST, DBNAME, USER, PASSWORD, PORT)
conn = psycopg2.connect(DBAuthorize)

## Fetch video watches (with metadata included)

In [7]:
cur = conn.cursor()

# video_watches_with_video_metadata doesn't exist as a table/view in this DB,
# so we join video_watches + video_metadata manually
query = """
    SELECT vw.*, vm.uploader, vm.description, vm.duration AS video_duration
    FROM video_watches vw
    LEFT JOIN video_metadata vm ON vw.video_id = vm.video_id
"""
cur.execute(query)
rows = cur.fetchall()
column_names = [desc[0] for desc in cur.description]

video_watches = pd.DataFrame(rows, columns=column_names)

cur.close()

print(f"video_watches rows: {len(video_watches):,}")
print(f"columns: {video_watches.columns.tolist()}")
video_watches.head()

video_watches rows: 8,071,423
columns: ['prolific_id', 'watch_datetime', 'video_id', 'next_video_start_time', 'time_to_next_video', 'time_from_previous_video', 'phase1_watch_or_skip', 'watch_duration_if_phase1_watch', 'phase1_session_stage', 'phase1_session_id', 'uploader', 'description', 'video_duration']


,prolific_id,watch_datetime,video_id,next_video_start_time,time_to_next_video,time_from_previous_video,phase1_watch_or_skip,watch_duration_if_phase1_watch,phase1_session_stage,phase1_session_id,uploader,description,video_duration
0,56b4b565b2de2a000d3316ba,2025-06-05 11:19:56,7512111089723739423,2025-06-05 12:00:27,2431.000,324.000,watch,0.000,session_start_and_end,5,galaxyofus,#tammy1000lbsisters #1000lbsisters #thousandpo...,84.000
1,56b4b565b2de2a000d3316ba,2025-06-05 16:04:34,7512428388397911342,2025-06-05 16:04:46,12.000,327.000,watch,12.000,session_start,11,None,None,NaN
2,56b4b565b2de2a000d3316ba,2025-06-05 16:32:51,7512493916277591326,2025-06-05 16:32:54,3.000,0.000,skip,0.000,mid_session,12,oseanfox,Day 65: Is it in you now to bear to hear the t...,33.000
3,56b4b565b2de2a000d3316ba,2025-06-06 01:48:55,7512621048484105518,2025-06-06 01:57:20,505.000,14.000,watch,0.000,session_end,25,kelloelle,#adhd,60.000
4,56b4b565b2de2a000d3316ba,2025-06-06 04:43:47,7512543275329408298,2025-06-06 04:43:57,10.000,6.000,watch,10.000,mid_session,27,texplays,"Relax, I'm blackish. #cod #texplays",60.000


In [8]:
video_watches.to_csv('video_watches.csv', index=False)
print("Saved video_watches.csv")

Saved video_watches.csv


## Identify TAB videos

Filter `video_watches_with_video_metadata` for tiktoktips uploader + take-a-break keywords.

In [9]:
# Since video_watches_with_video_metadata includes metadata columns,
# we can identify TAB videos directly from it
take_a_break_keywords = ["pause", "outside", "rest", "break", "snack", "present"]
tab_pattern = '|'.join(take_a_break_keywords)

# Filter for tiktoktips uploader with TAB keywords in description
uploader_col = 'uploader'  # adjust if column name differs
description_col = 'description'  # adjust if column name differs

# Get unique TAB videos
tab_videos = video_watches[
    (video_watches[uploader_col] == 'tiktoktips') &
    (video_watches[description_col].str.contains(tab_pattern, case=False, na=False))
][['video_id', uploader_col, description_col]].drop_duplicates(subset='video_id')

print(f"TAB videos found: {len(tab_videos)}")
tab_videos.to_csv('tab_videos.csv', index=False)
tab_videos

TAB videos found: 6


,video_id,uploader,description
1700753,7062479459739405615,tiktoktips,Give your body and mind some rest. You deserve...
3213454,6997542159805172997,tiktoktips,"Wait, why did this actually work? #BePresent @..."
3999038,6781608784990178566,tiktoktips,Pause your scrolling. Time for a night time sn...
6743934,7034264611809398022,tiktoktips,This is your sign to actually go to bed now. #...
7061182,7034264925996436741,tiktoktips,"Wait, why did this actually work? #BePresent @..."
7626495,7059993381795089710,tiktoktips,"Whatever you're feeling, come to the present w..."


## Fetch user sessions (filtered by cleaned dataset)

Uses `phase2_flattened_sessions` filtered to only include users in `cleaned_data_set_prolific_ids` and sessions with at least one video (`video_count > 0`). This excludes 31,162 empty sessions that were previously inflating session counts.

In [10]:
cur = conn.cursor()

query = """
    SELECT pfs.*
    FROM phase2_flattened_sessions pfs
    INNER JOIN cleaned_data_set_prolific_ids cdpi
        ON pfs.prolific_id = cdpi.prolific_id
    WHERE pfs.video_count > 0
"""
cur.execute(query)
rows = cur.fetchall()
column_names = [desc[0] for desc in cur.description]

user_sessions = pd.DataFrame(rows, columns=column_names)

cur.close()

print(f"user_sessions rows: {len(user_sessions):,}")
print(f"(31,162 sessions with video_count=0 excluded)")
print(f"columns: {user_sessions.columns.tolist()}")
user_sessions.head()

user_sessions rows: 231,439
(31,162 sessions with video_count=0 excluded)
columns: ['phase2_session_id', 'phase1_session_id', 'phase2_session_start', 'phase2_session_end', 'prolific_id', 'video_unique', 'video_count', 'session_duration_seconds']


,phase2_session_id,phase1_session_id,phase2_session_start,phase2_session_end,prolific_id,video_unique,video_count,session_duration_seconds
0,0,12176,2025-06-05 00:42:18,2025-06-05 00:46:51,56b4b565b2de2a000d3316ba,19.000,19.000,273.000
1,1,12177,2025-06-05 00:51:59,2025-06-05 00:54:31,56b4b565b2de2a000d3316ba,10.000,10.000,152.000
2,2,12178,2025-06-05 01:35:42,2025-06-05 01:37:39,56b4b565b2de2a000d3316ba,5.000,5.000,117.000
3,3,12179,2025-06-05 11:05:27,2025-06-05 11:14:32,56b4b565b2de2a000d3316ba,29.000,29.000,545.000
4,4,12180,2025-06-05 11:19:52,2025-06-05 11:19:56,56b4b565b2de2a000d3316ba,1.000,1.000,4.000


In [11]:
user_sessions.to_csv('user_sessions.csv', index=False)
print("Saved user_sessions.csv")

Saved user_sessions.csv


## Fetch session data (phase2_browsing_sessions, filtered by cleaned dataset)

In [12]:
cur = conn.cursor()

query = """
    SELECT pbs.*
    FROM phase2_browsing_sessions pbs
    INNER JOIN cleaned_data_set_prolific_ids cdpi
        ON pbs.prolific_id = cdpi.prolific_id
"""
cur.execute(query)
rows = cur.fetchall()
column_names = [desc[0] for desc in cur.description]

session_data = pd.DataFrame(rows, columns=column_names)

cur.close()

print(f"session_data rows: {len(session_data):,}")
print(f"columns: {session_data.columns.tolist()}")
session_data.head()

session_data rows: 8,071,423
columns: ['watch_datetime', 'video_id', 'phase2_session_id', 'phase2_session_stage', 'prolific_id']


,watch_datetime,video_id,phase2_session_id,phase2_session_stage,prolific_id
0,2025-06-05 00:42:18,7512242261631995182,0.000,session_start,56b4b565b2de2a000d3316ba
1,2025-06-05 00:42:19,7387193994180152619,0.000,mid_session,56b4b565b2de2a000d3316ba
2,2025-06-05 00:42:21,7512238457138269482,0.000,mid_session,56b4b565b2de2a000d3316ba
3,2025-06-05 00:42:24,7510332454817877270,0.000,mid_session,56b4b565b2de2a000d3316ba
4,2025-06-05 00:44:10,7512096854209842463,0.000,mid_session,56b4b565b2de2a000d3316ba


In [13]:
session_data.to_csv('session_data.csv', index=False)
print("Saved session_data.csv")

Saved session_data.csv


## Summary

In [14]:
print("=== Data Export Summary ===")
print(f"Database: fixed_tiktok_data_donation_dec_2025_cleaned_dataset")
print(f"\nFiles saved:")
print(f"  video_watches.csv — {len(video_watches):,} rows (from video_watches_with_video_metadata)")
print(f"  tab_videos.csv — {len(tab_videos)} TAB videos identified")
print(f"  user_sessions.csv — {len(user_sessions):,} rows (phase2_flattened_sessions filtered by cleaned_data_set_prolific_ids)")
print(f"  session_data.csv — {len(session_data):,} rows (phase2_browsing_sessions filtered by cleaned_data_set_prolific_ids)")
print(f"\nUnique users: {user_sessions['prolific_id'].nunique()}")

conn.close()
print("\nConnection closed.")

=== Data Export Summary ===
Database: fixed_tiktok_data_donation_dec_2025_cleaned_dataset

Files saved:
  video_watches.csv — 8,071,423 rows (from video_watches_with_video_metadata)
  tab_videos.csv — 6 TAB videos identified
  user_sessions.csv — 231,439 rows (phase2_flattened_sessions filtered by cleaned_data_set_prolific_ids)
  session_data.csv — 8,071,423 rows (phase2_browsing_sessions filtered by cleaned_data_set_prolific_ids)

Unique users: 150

Connection closed.
